# Tool Call Demo — all accessible providers

In [1]:
import asyncio, sys, datetime
from pathlib import Path
from typing import Any

sys.path.insert(0, str(Path().resolve()))
from unified_local_llm_server import LLMProviderPool
from unified_local_llm_server.llm_logger import AsyncLLMLogger

LOG_PATH = Path("test_logs/tool_call_demo.log")

## Tools

In [2]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_current_date_time",
            "description": "Returns the current date and time.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "add",
            "description": "Adds two numbers and returns the result.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number"},
                    "b": {"type": "number"},
                },
                "required": ["a", "b"],
            },
        },
    },
]

TOOL_REGISTRY = {
    "get_current_date_time": lambda _: datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "add": lambda args: str(args["a"] + args["b"]),
}


## Discover accessible providers

In [7]:
# Models known to support tool calling per provider
TOOL_MODELS = {
    "ollama":    "gpt-oss:20b",
    "lm_studio": "google/gemma-4-e4b",
    #"unsloth": "unsloth/gpt-oss-20b-GGUF@UD-Q4_K_XL",
}

server = LLMProviderPool("providers.example.yaml")

accessible = []
for name in server.provider_registry.names():
    try:
        status = await server.check_provider(name)
        if status.get("ok"):
            accessible.append(name)
            server.unload_all_models(name)
            print(f"  ✓ {name}")
        else:
            print(f"  ✗ {name} — {status}")
    except Exception as e:
        print(f"  ✗ {name} — {e}")
        

pairs = [(p, m) for p, m in TOOL_MODELS.items() if p in accessible]
print(f"\nTool-capable pairs: {pairs}")

  ✗ llama_cpp — {'provider': 'llama_cpp', 'server_url': 'http://127.0.0.1:8080', 'ok': False, 'kind': 'health', 'error': '<urlopen error [Errno 111] Connection refused>'}
  ✓ lm_studio
  ✓ ollama
  ✓ unsloth

Tool-capable pairs: [('ollama', 'gpt-oss:20b'), ('lm_studio', 'google/gemma-4-e4b')]


## Run tool call tests

In [6]:
logger = AsyncLLMLogger(LOG_PATH)

for provider, model in pairs:
    llm = server.load_model(provider, model, logger=logger)

    print(f"\n{'='*60}")
    print(f"  {provider} / {model}")
    print(f"{'='*60}")

    # Test 1: date/time
    result = await llm.call(
        messages=[{"role": "user", "content": "What is the current date and time? Use the tool."}],
        tools=TOOLS,
        tool_registry=TOOL_REGISTRY,
        think=True,
        options={"max_tokens": 200},
    )
    print(f"[date]   {result}")

    # Test 2: arithmetic
    result = await llm.call(
        messages=[{"role": "user", "content": "What is 1234 + 5678? Use the tool."}],
        tools=TOOLS,
        tool_registry=TOOL_REGISTRY,
        think=True,
        options={"max_tokens": 200},
    )
    print(f"[add]    {result}")

await logger.close()
print(f"\nLog: {LOG_PATH.resolve()}")


  ollama / gpt-oss:20b
[date]   The current date and time is: **2026-05-16 19:18:48**
[add]    The sum of 1234 and 5678 is **6912**.

  lm_studio / google/gemma-4-e4b
[date]   The current date and time is 2026-05-16 19:18:54.
[add]    The result of 1234 + 5678 is 6912.

Log: /home/ubn/Documents/projects/unified_local_llm_server/test_logs/tool_call_demo.log
